In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
import default_risk.config as cfg
import dtale
import logging



previous_application_df = pd.read_csv(cfg.PREVIOUS_APPLICATION)

log = logging.getLogger('werkzeug')


with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

In [ ]:
#files for the data dictionary
create_files_nulls_per_colmun(previous_application_df,"previous_application")

In [ ]:
eda_per_table_persisting_result_html(previous_application_df,schema,"previous_application",False)

In [ ]:
"""in the column of "FLAG_LAST_APPL_PER_CONTRACT" the description says: "Flag if it was last application for the previous contract. 
Sometimes by mistake of client or our clerk there could be more applications for one single contract, so i'm going to look for that potential errors"""

In [ ]:
previous_application_df["FLAG_LAST_APPL_PER_CONTRACT"].head()

In [ ]:
#there is no dups with SK_ID_PREV. si the check would be based per client because the description don't mention how a dup looks like.
previous_application_df.groupby("SK_ID_PREV").size().max()

In [ ]:
boolean_mask_for_flag= previous_application_df["FLAG_LAST_APPL_PER_CONTRACT"] == "Y"
clients_ids_more_than_one_flaged_as_final= boolean_mask_for_flag.groupby(previous_application_df["SK_ID_CURR"]).sum().loc[lambda g : g  > 1].index
rows_of_client_with_more_than_one=previous_application_df[previous_application_df["SK_ID_CURR"].isin(clients_ids_more_than_one_flaged_as_final)]
rows_of_client_with_more_than_one.head()
len(rows_of_client_with_more_than_one)

#and seems like have more than 1 previous application flagged as final is the most usual scenario.

In [ ]:
#so, the idea is check if the next solicitud of the same client, after a contract with that flag in "N", is similar. And that would be the "duplicates".
previous_application_df.sort_index
boolean_mask_for_flag_in_negative= previous_application_df["FLAG_LAST_APPL_PER_CONTRACT"] == "N" 
rows_applications_no_final= previous_application_df[boolean_mask_for_flag_in_negative] 
ids=rows_applications_no_final["SK_ID_CURR"]
ids_cutted= ids[:20]
clients_with_at_leas_a_no_final=previous_application_df[previous_application_df["SK_ID_CURR"].isin(ids_cutted)]
clients_with_at_leas_a_no_final.sort_values(["SK_ID_CURR","DAYS_DECISION"]).to_csv(cfg.DUMP_FROM_NOTEBOOKS /"pairs.csv")
dtale.show(clients_with_at_leas_a_no_final.sort_values(["SK_ID_CURR","DAYS_DECISION"]))
#in this manual check using dtale, i found that not always the applications flaged as no final have a re try but has been discover that have a very high correlation with rejection.
#so we will investigate they correlation with rejection and if treat it like a subset of rejection o two different sets.


In [ ]:
mask_not_last_in_day= previous_application_df["FLAG_LAST_APPL_PER_CONTRACT"] == "N" 
no_final= previous_application_df[mask_not_last_in_day] 
no_final_approved= no_final[no_final["NAME_CONTRACT_STATUS"] == "Approved"]
print(len(no_final_approved))
print(len(rows_applications_no_final) - len(no_final_approved)) 
dtale.show(rows_applications_no_final)
#Evidence of non existence a contract marked as not final and approved.

In [ ]:
mask_not_last_in_day= previous_application_df["NFLAG_LAST_APPL_IN_DAY"] == 0 
not_last_in_day_rows= previous_application_df[mask_not_last_in_day] 
one_flag_subset_of_another= (not_last_in_day_rows["FLAG_LAST_APPL_PER_CONTRACT"] == "Y").sum()
no_final_approved= (not_last_in_day_rows["NAME_CONTRACT_STATUS"] == "Approved")
print("with NFLAG_LAST_APPL_IN_DAY in 1 and approved: " + str(no_final_approved.sum()))
print("With flag NFLAG_LAST_APPL_IN_DAY in 1 and FLAG_LAST_APPL_PER_CONTRACT with Y: " + str(one_flag_subset_of_another))
not_last_in_day_rows[no_final_approved].head()

#so, there is cases where is not the last aplication of the day (NFLAG_LAST_APPL_IN_DAY == 0) but was the last one for the contract ("FLAG_LAST_APPL_PER_CONTRACT"] == "Y").
#And also cases where is not te last application of the day and the contract was "Approved".
#that means, one row are not subset of another, and seems the cases where the client apply for 2 diferents contracts in the same day.

In [ ]:
no_final_and_no_refused= no_final[no_final["NAME_CONTRACT_STATUS"] != "Refused"]
print(len(no_final_and_no_refused))
no_final_and_no_refused["NAME_CONTRACT_STATUS"].head()




"""so, every contract with that flag are marked as "refused" with exception of this 2, marked as canceled. So this provide enough evidence to treat
the rows with that flag as a subset of "refused" contract. The intention is create aggregation metrics of rejected contracts to modelate the diferences between 
the contracts that are approved to the client and the one that are refused. But in the visualization order by DAYS_DECISION seems like almost always there is a another application, similar to the one marked
as not final. So we can assume that if "FLAG_LAST_APPL_PER_CONTRACT" == "N" is a rejected contract and the client tried again with anothers numbers (you can see how sometimes re-apply with a smaller loan) or just
an error marked as refused. Would be interesting catch the diference when are legit retries and one are acepted and the other one no, but otherwise you are catching the same rejection twice, so we are not
dropping this as dups but are not directly used for agg metrics."""

In [ ]:
#in early analisis we detect a potencial incosistency in data
incosistency_mask= (previous_application_df["NAME_CONTRACT_STATUS"] == "Refused") & (previous_application_df["CODE_REJECT_REASON" ] == "XAP")
rows_flaged_as_refused_with_no_reason= previous_application_df[incosistency_mask]
rows_flaged_as_refused_with_no_reason.head()
#but is not final ("FLAG_LAST_APPL_PER_CONTRACT" == "N"). So, will not affect nothing let it in the dataset. 

In [7]:
#during the construction of data dictionary we detect almost perfect correlation between the missing values in AMT_ANNUITY and CNT_PAYMENT so we decide to check for exceptions. 
#in further investigations we found that annuity and cnt_payment are relationaned by a formula, so the expectable is both being null at the same time, and being a "Consumer loan" we can't figure out
# other explaniation than data corruption, so, in order to maintain the data integrity, we decide to drop it. 
mask_same_nulls= (previous_application_df["AMT_ANNUITY"].isnull()) & (previous_application_df["CNT_PAYMENT"].notna())
dtale.show(previous_application_df[mask_same_nulls])    

In [15]:
previous_application_df[ previous_application_df["RATE_DOWN_PAYMENT"] < 0].head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
368107,1284109,350530,Consumer loans,7866.090,71580.60,71581.5,-0.90,71580.60,SATURDAY,13,...,Connectivity,10.0,low_normal,POS mobile without interest,365243.0,-459.0,-189.0,-189.0,-185.0,0.0
1519595,1817983,133068,Consumer loans,3595.545,32719.05,32719.5,-0.45,32719.05,SUNDAY,13,...,Connectivity,10.0,low_normal,POS mobile without interest,365243.0,-430.0,-160.0,-430.0,-415.0,0.0
